# Satellite-Based Chlorophyll and NDCI Extraction for Lake Monitoring

This notebook extracts chlorophyll-a concentration estimates and Normalized Difference Chlorophyll Index (NDCI) values from multiple satellite platforms (MODIS-Terra, MODIS-Aqua, and Sentinel-2, as well as Landsat for reference) for specified lake locations.

## Overview

- Purpose: Generate time series of chlorophyll indices from satellite imagery
- Study Areas: Detroit Lake and Upper Klamath Lake
- Satellite Sensors: MODIS (Terra and Aqua), Sentinel-2.
- Reference Satellite Sensors: Landsat 5/7/8 for reference.
- Output: CSV files with date-stamped chlorophyll/NDCI values

## Key Features

- Cloud masking using QA bands
- Water detection using NDWI (Normalized Difference Water Index)
- 500 m x 500 m spatial averaging around lake centers
- Multi-sensor harmonization for long-term monitoring

In [1]:
"""
Initialize Google Earth Engine (GEE) connection and authentication.

This cell sets up the GEE Python API for accessing satellite imagery collections.
Authentication is required on first use or when credentials expire.
"""

import ee
import math
import pandas as pd

# Authenticate and initialize GEE with your registered project
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')  # Replace with your GEE Cloud Project ID

## Landsat Processing (30m resolution)

### Overview
Landsat provides the longest continuous Earth observation record (1984-present) with consistent 30m spatial resolution. This section processes Landsat 5 TM, 7 ETM+, and 8 OLI/TIRS imagery to derive NDCI values.

### Key Processing Steps:
1. **Cloud Masking**: Uses QA_PIXEL band to remove clouds, shadows, snow, and cirrus
2. **Water Masking**: Applies NDWI threshold to isolate water pixels
3. **NDCI Calculation**: (Band 5 - Band 4) / (Band 5 + Band 4) for vegetation/algae detection
4. **Spatial Averaging**: Computes mean NDCI over 500m x 500m region

### Band Mappings:
- **Landsat 5/7**: Band 4 = Red (0.63-0.69 μm), Band 5 = NIR (0.76-0.90 μm)
- **Landsat 8**: Band 4 = Red (0.64-0.67 μm), Band 5 = NIR (0.85-0.88 μm)

In [2]:
# ============================================================================
# LANDSAT NDCI EXTRACTION FOR LAKE MONITORING
# ============================================================================

# ------------ USER CONFIGURATION --------------------------------------------
# Define lake locations and output filenames
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         export_id='Detroit_Landsat_NDCI_500m'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         export_id='Klamath_Landsat_NDCI_500m')
]

# Temporal range for analysis
start_date, end_date = '2011-01-01', '2025-12-31'

# Spatial buffer around lake center point (250m gives 500m x 500m box)
half_size = 250  # metres

# ------------ SATELLITE COLLECTIONS -----------------------------------------
# Merge Landsat 5, 7, and 8 Level-2 Surface Reflectance collections
landsat = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')  # Landsat 5 TM
           .merge(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2'))  # Landsat 7 ETM+
           .merge(ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')))  # Landsat 8 OLI

# ------------ PREPROCESSING FUNCTIONS ---------------------------------------
def mask_clouds(img):
    """
    Apply cloud mask using QA_PIXEL band bit flags.
    
    Bit positions in QA_PIXEL:
    - Bit 3: Cloud
    - Bit 4: Cloud shadow
    - Bit 5: Snow
    - Bit 2: Cirrus (Landsat 8 only)
    """
    qa = img.select('QA_PIXEL')
    mask = (qa.bitwiseAnd(1 << 3).eq(0)      # No cloud
            .And(qa.bitwiseAnd(1 << 4).eq(0))  # No cloud shadow
            .And(qa.bitwiseAnd(1 << 5).eq(0))  # No snow
            .And(qa.bitwiseAnd(1 << 2).eq(0))) # No cirrus
    return img.updateMask(mask)

def add_water_mask(img):
    """
    Apply water mask using NDWI (Normalized Difference Water Index).
    NDWI = (Green - NIR) / (Green + NIR)
    Water pixels have NDWI > 0
    """
    bands = img.bandNames()
    
    # Select NIR band based on Landsat sensor
    # Landsat 8: Band 5 (NIR), Landsat 5/7: Band 4 (NIR)
    nir = ee.Algorithms.If(
            bands.contains('SR_B5'),  # Landsat 8 OLI/OLI-2
            'SR_B5',
            'SR_B4'                    # Landsat 5 TM and 7 ETM+
          )
    
    # Calculate NDWI using Green (B3) and NIR bands
    ndwi = img.normalizedDifference(['SR_B3', ee.String(nir)])
    return img.updateMask(ndwi.gt(0))

def drop_low_edge(img, edge_band, thresh=0.002):
    """
    Remove pixels with very low red-edge reflectance (likely deep shadows).
    
    Args:
        img: Input image
        edge_band: Name of red-edge band
        thresh: Minimum reflectance threshold (default 0.002)
    """
    # Apply scaling factors to convert DN to reflectance
    scale = 1e-4 if edge_band == 'B5' else 2.75e-5
    edge = img.select(edge_band).multiply(scale).add(-0.2)
    return img.updateMask(edge.gt(thresh))

def add_ndci(img):
    """
    Calculate NDCI (Normalized Difference Chlorophyll Index).
    NDCI = (NIR - Red) / (NIR + Red)
    
    Higher NDCI values indicate higher chlorophyll/algae content.
    """
    # Convert to surface reflectance (scale + offset)
    sr = img.select(['SR_B4', 'SR_B5']).multiply(2.75e-5).add(-0.2)
    
    # Calculate NDCI: (B5 - B4) / (B5 + B4)
    ndci = sr.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDCI')
    return img.addBands(ndci)

def img_to_feat(img, roi, tag):
    """
    Convert image to feature with mean NDCI value over ROI.
    
    Args:
        img: Input image with NDCI band
        roi: Region of interest geometry
        tag: Sensor identifier tag
    
    Returns:
        Feature with date, NDCI value, and sensor tag
    """
    # Calculate mean NDCI within ROI
    mean = img.select('NDCI').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=30,  # Landsat pixel size
        maxPixels=1e9).get('NDCI')
    
    # Return feature only if valid NDCI value exists
    return ee.Algorithms.If(
        mean,
        ee.Feature(None, {
            'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'ndci': mean,
            'sensor': tag}),
        None)

def preprocess_landsat(img):
    """
    Complete preprocessing pipeline for Landsat imagery.
    """
    img = mask_clouds(img)      # Apply QA_PIXEL cloud mask
    img = add_water_mask(img)   # Keep water pixels only (NDWI > 0)
    
    # Remove extremely dark pixels (deep shadows)
    edge = img.select('SR_B5').multiply(2.75e-5).add(-0.2) \
            if img.bandNames().contains('SR_B5') \
            else img.select('SR_B4').multiply(2.75e-5).add(-0.2)
    img = img.updateMask(edge.gt(0.002))
    
    return img

# ------------ PROCESS EACH LAKE ---------------------------------------------
for lake in lakes:
    # Define ROI as 500m x 500m box centered on lake coordinates
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(half_size).bounds()
    
    # Process Landsat collection
    series = (landsat
              .filterDate(start_date, end_date)
              .filterBounds(roi)
              .map(preprocess_landsat)
              .map(add_ndci)
              .map(lambda img: img_to_feat(img, roi, lake['name'] + '_Landsat'),
                   dropNulls=True))
    
    # Print scene count
    print(lake['name'],
          'valid Landsat scenes =',
          series.aggregate_count('ndci').getInfo())
    
    # Export to CSV (client-side processing)
    rows = series.getInfo()['features']
    records = [f['properties'] for f in rows]
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['export_id'] + '.csv', index=False)
    
    # Alternative: Export to Google Drive (server-side)
    # ee.batch.Export.table.toDrive(
    #     collection=series,
    #     description=lake['export_id'],
    #     fileFormat='CSV'
    # ).start()

Detroit valid Landsat scenes = 511
UpperKlamath valid Landsat scenes = 694


## Sentinel-2 Processing (10m resolution)

### Overview
Sentinel-2 provides high spatial (10-20m) and temporal (5-day revisit) resolution multispectral imagery since 2015. The constellation consists of two satellites (2A and 2B) offering 13 spectral bands optimized for vegetation and water monitoring.

### Key Processing Steps:
1. **Cloud Masking**: Uses QA60 band to identify and remove clouds and cirrus
2. **Water Detection**: NDWI thresholding to isolate water bodies
3. **NDCI Calculation**: Uses red (665 nm) and red-edge (705 nm) bands
4. **Quality Control**: Removes edge artifacts and very dark pixels

### Advantages over Landsat:
- Higher spatial resolution (10m vs 30m)
- Red-edge band specifically designed for chlorophyll detection
- More frequent revisits (5 days vs 16 days)

In [3]:
# ============================================================================
# SENTINEL-2 NDCI EXTRACTION FOR LAKE MONITORING
# ============================================================================

# ------------ USER CONFIGURATION --------------------------------------------
# Define lake locations and output filenames
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         export_id='Detroit_S2_NDCI_500m'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         export_id='Klamath_S2_NDCI_500m')
    # Add more lakes here if needed
]

# Temporal range (Sentinel-2 available from July 2015)
start_date, end_date = '2015-07-01', '2025-12-31'

# Spatial buffer for 500m x 500m ROI
half_size_m = 250  # metres

# ------------ SATELLITE COLLECTION ------------------------------------------
# Sentinel-2 Level-2A Surface Reflectance (harmonized collection)
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

# ------------ PREPROCESSING FUNCTIONS ---------------------------------------
def mask_s2(img):
    """
    Apply cloud and cirrus mask using QA60 band.
    
    QA60 bit flags:
    - Bit 10: Opaque clouds
    - Bit 11: Cirrus clouds
    
    Also removes edge artifacts where B8A (NIR narrow) = 0
    """
    qa = img.select('QA60')
    cloud = qa.bitwiseAnd(1 << 10).neq(0)   # Opaque cloud flag
    cirrus = qa.bitwiseAnd(1 << 11).neq(0)  # Cirrus cloud flag
    mask = cloud.Or(cirrus).Not()           # Clear pixels
    
    # Remove edge stripes where B8A = 0
    return img.updateMask(mask)\
              .updateMask(img.select('B8A').gt(0))

def add_water_mask_s2(img):
    """
    Apply water mask using NDWI.
    
    NDWI = (Green - NIR) / (Green + NIR)
    Uses B3 (Green, 560nm) and B8 (NIR, 842nm)
    """
    ndwi = img.normalizedDifference(['B3', 'B8'])
    return img.updateMask(ndwi.gt(0))

def add_ndci(img):
    """
    Calculate NDCI using red and red-edge bands.
    
    NDCI = (Red-edge - Red) / (Red-edge + Red)
    - B4: Red band (665 nm) - sensitive to chlorophyll absorption
    - B5: Red-edge band (705 nm) - sensitive to vegetation/algae
    
    Scale factor for Sentinel-2 L2A = 1e-4
    """
    # Convert to surface reflectance
    sr = img.select(['B4', 'B5']).multiply(1e-4)
    red = sr.select('B4')   # 665 nm
    edge = sr.select('B5')  # 705 nm (red-edge)
    
    # Calculate NDCI
    ndci = edge.subtract(red)\
               .divide(edge.add(red))\
               .rename('NDCI')
    return img.addBands(ndci)

def preprocess_s2(img):
    """
    Complete preprocessing pipeline for Sentinel-2 imagery.
    
    Steps:
    1. Cloud/cirrus masking
    2. Water detection
    3. Remove very dark pixels
    """
    img = mask_s2(img)            # QA60 cloud/cirrus mask
    img = add_water_mask_s2(img)  # NDWI > 0 for water
    
    # Remove very dark red-edge pixels (shadows, poor quality)
    edge = img.select('B5').multiply(1e-4)
    img = img.updateMask(edge.gt(0.002))
    
    return img

def img_to_feature(img, roi, tag):
    """
    Convert image to feature with mean NDCI over ROI.
    
    Args:
        img: Sentinel-2 image with NDCI band
        roi: Region of interest geometry
        tag: Sensor identifier tag
    
    Returns:
        Feature with date, NDCI value, and sensor tag
    """
    # Calculate mean NDCI within ROI
    mean = img.select('NDCI').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,  # Sentinel-2 pixel size (10m for visible/NIR bands)
        maxPixels=1e9
    ).get('NDCI')
    
    # Return feature only if valid NDCI exists
    return ee.Algorithms.If(
        mean,
        ee.Feature(None, {
            'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'ndci': mean,
            'sensor': tag
        }),
        None)

# ------------ PROCESS EACH LAKE ---------------------------------------------
for lake in lakes:
    # Define ROI as 500m x 500m box centered on lake
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(half_size_m).bounds()
    
    # Process Sentinel-2 collection
    series = (s2.filterDate(start_date, end_date)
               .filterBounds(roi)
               .map(preprocess_s2)
               .map(add_ndci)
               .map(lambda img: img_to_feature(img, roi, lake['name'] + '_S2'),
                    dropNulls=True))
    
    # Print scene count
    print(lake['name'],
          'valid Sentinel-2 scenes =',
          series.aggregate_count('ndci').getInfo())
    
    # Export to CSV (client-side processing)
    rows = series.getInfo()['features']
    records = [f['properties'] for f in rows]
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['export_id'] + '.csv', index=False)
    
    # Alternative: Export to Google Drive (server-side)
    # ee.batch.Export.table.toDrive(
    #     collection=series,
    #     description=lake['export_id'],
    #     fileFormat='CSV'
    # ).start()

Detroit valid Sentinel-2 scenes = 286
UpperKlamath valid Sentinel-2 scenes = 1556


## MODIS Processing (500m resolution)

### Overview
MODIS (Moderate Resolution Imaging Spectroradiometer) provides daily global coverage at 500m resolution from two satellites: Terra (morning overpass) and Aqua (afternoon overpass). This section derives chlorophyll-a concentrations using an empirical green-to-red ratio algorithm.

### Chlorophyll Algorithm:
The algorithm uses the ratio of green (555 nm) to red (645 nm) reflectance:

$$
log_{10}(Chl-a) = 1.70 × log_{10}(R_{rs,555}/R_{rs,645}) + 1.54
$$

Where:
- $R_{rs}$ = Remote sensing reflectance (sr/π)
- Coefficients calibrated for inland waters

### Key Features:
- **Daily coverage**: Maximizes temporal resolution for trend analysis
- **Dual satellites**: Terra (AM) and Aqua (PM) provide two observations per day
- **Long record**: Continuous data since 2000
- **Trade-off**: Lower spatial resolution (500m) but higher temporal frequency

In [4]:
# ============================================================================
# MODIS CHLOROPHYLL-A EXTRACTION FOR LAKE MONITORING
# ============================================================================
"""
Generate daily chlorophyll-a time series from MODIS Terra and Aqua satellites.
Each lake receives two separate CSV files (one per satellite) with Chl-a estimates.
"""

# ------------ USER CONFIGURATION --------------------------------------------
# Define lake locations and output filenames
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         terra_export='Detroit_MODIS_Terra_500m_Chl_singlePixel',
         aqua_export='Detroit_MODIS_Aqua_500m_Chl_singlePixel'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         terra_export='Klamath_MODIS_Terra_500m_Chl_singlePixel',
         aqua_export='Klamath_MODIS_Aqua_500m_Chl_singlePixel')
    # Add more lakes here if desired
]

# ------------ GLOBAL SETTINGS -----------------------------------------------
# Temporal range for analysis
start_date = '2011-01-01'
end_date = '2025-12-31'

# Chlorophyll algorithm coefficients (green:red ratio method)
# Based on empirical calibration for inland waters
offset = 1.54  # Y-intercept (use 0.96 for UKL low-end adjust if needed)
slope = 1.70   # Slope coefficient

# ------------ PREPROCESSING FUNCTIONS ---------------------------------------
def mask_light_cloud(img):
    """
    Apply light cloud mask using state_1km QA band.
    
    QA bit flags:
    - Bit 10: Cloud state
    - Bit 12: Snow/ice flag
    
    This is a lighter mask than strict cloud detection to preserve more data.
    """
    qa = img.select('state_1km')
    cloud = qa.bitwiseAnd(1 << 10).neq(0)
    snow = qa.bitwiseAnd(1 << 12).neq(0)
    return img.updateMask(cloud.Not()).updateMask(snow.Not())

def add_chlorophyll(img):
    """
    Calculate chlorophyll-a concentration using green-to-red ratio algorithm.
    
    Algorithm:
    1. Convert surface reflectance to remote sensing reflectance (Rrs = sr/π)
    2. Calculate log10 ratio of green (555nm) to red (645nm)
    3. Apply linear regression: log10(Chl) = slope × log10(ratio) + offset
    4. Convert from log10 to linear scale
    
    Bands used:
    - Band 4 (sur_refl_b04): Green (545-565 nm)
    - Band 1 (sur_refl_b01): Red (620-670 nm)
    
    Returns:
        Image with added 'chlor_a' band in µg/L
    """
    # Convert to surface reflectance (scale factor = 1e-4)
    sr555 = img.select('sur_refl_b04').multiply(1e-4)  # Green (555 nm)
    sr645 = img.select('sur_refl_b01').multiply(1e-4)  # Red (645 nm)
    
    # Convert to remote sensing reflectance (divide by π)
    rrs555 = sr555.divide(ee.Number(math.pi))
    rrs645 = sr645.divide(ee.Number(math.pi))
    
    # Calculate log10 of green/red ratio
    log_ratio = rrs555.divide(rrs645).log10()
    
    # Apply empirical algorithm: log10(Chl) = slope × log_ratio + offset
    log10_chl = log_ratio.multiply(slope).add(offset)
    
    # Convert from log10 to linear scale (µg/L)
    chl = ee.Image(10).pow(log10_chl).rename('chlor_a')
    
    return img.addBands(chl)

def build_series(col_id, sensor_tag, pt):
    """
    Build chlorophyll time series for a specific MODIS collection and location.
    
    Args:
        col_id: GEE collection ID (MOD09GA for Terra, MYD09GA for Aqua)
        sensor_tag: Label for the sensor ('Terra' or 'Aqua')
        pt: Point geometry for sampling location
    
    Returns:
        Feature collection with date, chlorophyll value, and sensor tag
    """
    # Process MODIS collection
    collection = (ee.ImageCollection(col_id)
                  .filterDate(start_date, end_date)
                  .filterBounds(pt)
                  .map(mask_light_cloud)
                  .map(add_chlorophyll)
                  .select('chlor_a'))
    
    def img_to_feature(img):
        """
        Extract chlorophyll value at point location.
        
        Samples single 500m pixel at lake center point.
        Returns None if no valid data available.
        """
        # Sample single pixel at point location
        fc = img.sample(region=pt,
                        scale=500,      # MODIS pixel size
                        numPixels=1,    # Single pixel
                        geometries=False)
        
        # Return feature only if valid data exists
        return ee.Algorithms.If(
            fc.size().gt(0),
            ee.Feature(None, {
                'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
                'chl': fc.first().get('chlor_a'),
                'sensor': sensor_tag
            }),
            None)
    
    return collection.map(img_to_feature, dropNulls=True)

# ------------ PROCESS EACH LAKE ---------------------------------------------
for lake in lakes:
    # Define point geometry for lake center
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    
    # Build time series for both Terra and Aqua
    terra_series = build_series('MODIS/061/MOD09GA', 'Terra', center)
    aqua_series = build_series('MODIS/061/MYD09GA', 'Aqua', center)
    
    # Print data availability
    print(lake['name'], 'Terra rows =',
          terra_series.aggregate_count('chl').getInfo())
    print(lake['name'], 'Aqua rows =',
          aqua_series.aggregate_count('chl').getInfo())
    
    # Export Terra data to CSV
    rows = terra_series.getInfo()['features']
    records = [f['properties'] for f in rows]
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['terra_export'] + '.csv', index=False)
    
    # Export Aqua data to CSV
    rows = aqua_series.getInfo()['features']
    records = [f['properties'] for f in rows]
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['aqua_export'] + '.csv', index=False)
    
    # Alternative: Export to Google Drive (server-side)
    # ee.batch.Export.table.toDrive(
    #     collection=terra_series,
    #     description=lake['terra_export'],
    #     fileFormat='CSV'
    # ).start()
    #
    # ee.batch.Export.table.toDrive(
    #     collection=aqua_series,
    #     description=lake['aqua_export'],
    #     fileFormat='CSV'
    # ).start()

Detroit Terra rows = 1841
Detroit Aqua rows = 1867
UpperKlamath Terra rows = 2682
UpperKlamath Aqua rows = 2587
